In [ ]:
import sys
import os
from langchain.chat_models import init_chat_model

from dotenv import load_dotenv

load_dotenv(override=True)

In [ ]:
LLM_MODEL = os.getenv("LLM_MODEL")
LLM_BASE_URL=os.getenv("LLM_BASE_URL")
LLM_API_KEY=os.getenv("LLM_API_KEY")
LLM_TEMPERATURE=os.getenv("LLM_TEMPERATURE")

# vLLM 모델 인스턴스 생성
llm = init_chat_model(
    "openai:" + LLM_MODEL,
    temperature=LLM_TEMPERATURE,
    base_url=LLM_BASE_URL,
    api_key=LLM_API_KEY
)

In [ ]:
import os
from pathlib import Path

def extract_text_from_pdf(pdf_path: str) -> str:
    """
    PDF 파일에서 텍스트를 추출하는 함수
    
    Args:
        pdf_path: PDF 파일 경로 (상대 경로 또는 절대 경로)
    
    Returns:
        추출된 텍스트 문자열
    
    Raises:
        FileNotFoundError: PDF 파일을 찾을 수 없을 때
        ImportError: 필요한 PDF 라이브러리가 설치되지 않았을 때
    """
    # 파일 경로 확인 및 절대 경로로 변환
    pdf_path = Path(pdf_path)
    if not pdf_path.is_absolute():
        # 노트북 위치 기준 상대 경로 처리
        # 노트북은 asset_ai_portal/tests 폴더에 있고, documents는 20_code_test 루트에 있음
        current_dir = Path.cwd()
        
        # asset_ai_portal/tests에서 실행 중이면 상위로 두 번 이동 (20_code_test 루트)
        if current_dir.name == 'tests' and current_dir.parent.name == 'asset_ai_portal':
            project_root = current_dir.parent.parent  # tests -> asset_ai_portal -> 20_code_test
        elif current_dir.name == 'asset_ai_portal':
            project_root = current_dir.parent  # asset_ai_portal -> 20_code_test
        else:
            # 20_code_test에서 실행 중이면 그대로 사용
            project_root = current_dir
        
        pdf_path = project_root / pdf_path
    
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF 파일을 찾을 수 없습니다: {pdf_path}")
    
    # 여러 PDF 라이브러리 시도 (우선순위 순)
    # 1. pypdf (가장 가벼움)
    try:
        from pypdf import PdfReader
        reader = PdfReader(str(pdf_path))
        text_parts = []
        for page in reader.pages:
            text_parts.append(page.extract_text())
        return "\n".join(text_parts)
    except ImportError:
        pass
    
    # 2. pdfplumber (표 추출에 유리)
    try:
        import pdfplumber
        text_parts = []
        with pdfplumber.open(str(pdf_path)) as pdf:
            for page in pdf.pages:
                text = page.extract_text()
                if text:
                    text_parts.append(text)
        return "\n".join(text_parts)
    except ImportError:
        pass
    
    # 3. PyPDF2 (구버전 호환)
    try:
        import PyPDF2
        with open(pdf_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            text_parts = []
            for page in reader.pages:
                text_parts.append(page.extract_text())
            return "\n".join(text_parts)
    except ImportError:
        pass
    
    # 모든 라이브러리가 없으면 에러
    raise ImportError(
        "PDF 텍스트 추출을 위한 라이브러리가 설치되지 않았습니다. "
        "다음 중 하나를 설치해주세요: pypdf, pdfplumber, PyPDF2\n"
        "설치 명령: pip install pypdf 또는 pip install pdfplumber"
    )


# 사용 예시: 대신.pdf 파일 텍스트 추출
pdf_file_path = "documents/sample_overseas_settlement/SB(Bernstein).pdf"
try:
    pdf_text = extract_text_from_pdf(pdf_file_path)
    print(f"✅ PDF 텍스트 추출 완료 ({len(pdf_text)} 문자)")
    print("\n" + "="*80)
    print("추출된 텍스트:")
    print("="*80)
    print(pdf_text[:1000])  # 처음 1000자만 미리보기
    if len(pdf_text) > 1000:
        print(f"\n... (총 {len(pdf_text)} 문자 중 처음 1000자만 표시)")
except Exception as e:
    print(f"❌ 오류 발생: {e}")


In [ ]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

# 메시지 객체 생성
system_msg = SystemMessage("당신은 자산운용사에서 해외거래체결 확인을 담당하는 오퍼레이터 입니다.")
human_msg = HumanMessage(f"""
아래는 브로커가 보내온 해외거래체결내역 메일입니다.
메일 내용을 분석하여 해외거래체결내역 대사 업무를 위한 거래 체결 정보를 수집하세요.

** 반드시 지켜야 할 중요한 사항 **
1. 모든 종목을 전부 수집하세요.(주요 종목만 수집하면 안됩니다.)

### 해외거래체결내역 메일 내용 ###
{pdf_text}
""")

# 채팅 모델과 함께 사용
messages = [system_msg, human_msg]
response = llm.invoke(messages)  # AIMessage 반환

In [ ]:
from IPython.display import Markdown, display

# LLM 응답을 마크다운 형식으로 보기 좋게 표시
if 'response' in locals():
    display(Markdown(response.content))
    
    # 추가 정보 (토큰 사용량 등)를 표시
    if hasattr(response, 'response_metadata') and response.response_metadata:
        metadata = response.response_metadata
        if 'token_usage' in metadata:
            print("\n---")
            print("**토큰 사용량:**")
            print(f"- 입력 토큰: {metadata['token_usage'].get('prompt_tokens', 'N/A')}")
            print(f"- 출력 토큰: {metadata['token_usage'].get('completion_tokens', 'N/A')}")
            print(f"- 총 토큰: {metadata['token_usage'].get('total_tokens', 'N/A')}")
else:
    print("⚠️ 'response' 변수를 찾을 수 없습니다. 먼저 LLM을 호출해주세요.")
